# Advanced lab 5 — One OpenTelemetry trace in Foundry and MLflow

The correct architecture is exporter fan-out, not backend synchronization: one OpenTelemetry provider creates one trace and sends each finished span to Azure Monitor/Application Insights and an MLflow OTLP receiver. Foundry remains the operational agent view. **MLflow is the authoritative assurance evidence plane** for reviewed traces, EvaluationDatasets, evaluation runs, lineage, and Feedback / Assessments.

Sources: [Foundry client-side tracing](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-client-side), [MLflow OTLP ingestion](https://mlflow.org/docs/latest/genai/tracing/opentelemetry/ingest/), and [Unity Catalog trace storage](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/trace-unity-catalog).

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path
from urllib.parse import urlsplit
from uuid import UUID

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
session.safe_summary()

```text
Foundry agent / application
          |
 optional hosted protocol parent (validate live)
          |
   one application OTel provider and trace ID
          |
     +----+----+
     |         |
Azure Monitor  OTLP/HTTP
     |         |
Foundry UI    MLflow / Databricks UC
operations    authoritative assurance evidence
```

Do not independently enable two instrumentation owners; that can create duplicate spans or unrelated trace IDs. Configure once in a fresh kernel, keep message-content capture off, and restart the kernel before changing tracing mode. The current preview SDK also requires `AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true` before instrumentation; the connected cell fails closed when that explicit feature gate is absent.

Privacy follows the producer. SDK-owned client spans use this application's provider and `AIProjectInstrumentor` content/context/baggage switches. Foundry-native or hosted spans are produced by the managed protocol runtime and follow its telemetry policy. Disabling content in one owner does not globally redact exception text, identifiers, tool definitions, or custom attributes from the other. Optional provider-supported reasoning is diagnostics only; never request or reconstruct hidden chain-of-thought.

For certified Agent Framework Core 1.12.1, the canonical application subtree is `invoke_agent` with repeated `chat` and `execute_tool` **sibling children**. TOOL is not nested under the chat span that requested it. A hosted protocol runtime can add another root, and managed MLflow can translate native types, so validate both the outer parent and rendered mapping live.

The connected path below is only a **dual-export/correlation smoke test** using the Foundry project client. It does not invoke Agent Framework or execute a real tool loop. The certified Agent Framework tool hierarchy therefore remains authenticated live validation in an approved agent application; this lab does not add another dependency or pretend that a one-word Responses call proved that hierarchy.

In [ ]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import (
    InMemorySpanExporter,
)

foundry_probe = InMemorySpanExporter()
mlflow_probe = InMemorySpanExporter()
offline_provider = TracerProvider()
offline_provider.add_span_processor(SimpleSpanProcessor(foundry_probe))
offline_provider.add_span_processor(SimpleSpanProcessor(mlflow_probe))
offline_tracer = offline_provider.get_tracer("foundry-curriculum.dual-export")

with offline_tracer.start_as_current_span("invoke_agent") as agent_span:
    agent_span.set_attribute("gen_ai.operation.name", "invoke_agent")
    agent_span.set_attribute("gen_ai.agent.name", "synthetic-agent")
    agent_span.set_attribute("session.id", "synthetic-conversation-001")
    agent_span.set_attribute("aai.measurement_source", "synthetic_offline_topology")
    emitted_trace_id = agent_span.get_span_context().trace_id
    with offline_tracer.start_as_current_span("chat") as first_chat:
        first_chat.set_attribute("gen_ai.operation.name", "chat")
        first_chat.set_attribute("gen_ai.request.model", "synthetic-model")
    with offline_tracer.start_as_current_span("execute_tool") as tool_span:
        tool_span.set_attribute("gen_ai.operation.name", "execute_tool")
        tool_span.set_attribute("gen_ai.tool.name", "synthetic_lookup")
    with offline_tracer.start_as_current_span("chat") as final_chat:
        final_chat.set_attribute("gen_ai.operation.name", "chat")
        final_chat.set_attribute("gen_ai.request.model", "synthetic-model")

offline_provider.force_flush()


def exported_shape(exporter):
    return {
        (
            item.name,
            item.context.trace_id,
            item.context.span_id,
            item.parent.span_id if item.parent else None,
        )
        for item in exporter.get_finished_spans()
    }


foundry_shape = exported_shape(foundry_probe)
mlflow_shape = exported_shape(mlflow_probe)
assert foundry_shape == mlflow_shape
assert {trace_id for _name, trace_id, _span_id, _parent_id in foundry_shape} == {
    emitted_trace_id
}
invoke_span = next(
    item for item in foundry_probe.get_finished_spans() if item.name == "invoke_agent"
)
native_children = [
    item
    for item in foundry_probe.get_finished_spans()
    if item.name in {"chat", "execute_tool"}
]
assert [item.name for item in native_children] == [
    "chat",
    "execute_tool",
    "chat",
]
assert all(
    item.parent.span_id == invoke_span.context.span_id for item in native_children
)
{
    "trace_id_hex": f"{emitted_trace_id:032x}",
    "exporters": 2,
    "same_span_ids_and_parents": True,
    "measurement_source": "synthetic_offline_topology",
}

In [ ]:
backend_contracts = {
    "foundry": {
        "receiver": "connected Application Insights via Azure Monitor exporter",
        "role": "operational diagnosis and Foundry trace view",
        "prerequisites": [
            "Application Insights linked to the project",
            "Log Analytics Reader for the viewer",
            "agent reference plus conversation correlation",
        ],
        "expected_delay": "typically 2-5 minutes for client-side traces",
    },
    "oss_mlflow": {
        "transport": "OTLP/HTTP only",
        "path": "/v1/traces",
        "routing_header": "x-mlflow-experiment-id",
        "storage": "SQL backend required",
    },
    "databricks_uc": {
        "transport": "managed Databricks OTLP/HTTP endpoint",
        "path": "/api/2.0/otel/v1/traces",
        "routing_header": "X-Databricks-UC-Table-Name",
        "direct_auth": "short-lived token acquired and renewed by the runtime",
        "long_running_auth": ("collector/gateway owns token acquisition and refresh"),
        "warning": (
            "never freeze a static Databricks bearer token into an agent version"
        ),
    },
}
backend_contracts

In [ ]:
RUN_DUAL_EXPORT = False


def configure_connected_dual_export():
    import httpx
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.telemetry import AIProjectInstrumentor
    from azure.identity import DefaultAzureCredential
    from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
    from opentelemetry import trace
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import (
        OTLPSpanExporter,
    )
    from opentelemetry.sdk.resources import Resource
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import BatchSpanProcessor

    if os.environ.get("AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING", "").lower() != "true":
        raise RuntimeError(
            "Set AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true before "
            "enabling the preview Foundry instrumentor."
        )

    if not session.trace_ready:
        raise RuntimeError(
            "Configure agent name, version, and trace-correlation ID first."
        )
    endpoint = os.environ.get("OTEL_EXPORTER_OTLP_TRACES_ENDPOINT", "").strip()
    headers_present = bool(
        os.environ.get("OTEL_EXPORTER_OTLP_TRACES_HEADERS", "").strip()
    )
    parsed = urlsplit(endpoint)
    if parsed.scheme not in {"http", "https"} or not parsed.hostname:
        raise RuntimeError("Configure a valid OTLP/HTTP traces endpoint externally.")
    if "/api/2.0/otel/v1/traces" in parsed.path:
        raise RuntimeError(
            "This generic exporter reads static headers; route managed "
            "Databricks export through the approved renewable-auth gateway."
        )
    if not headers_present:
        raise RuntimeError("Configure OTLP routing/auth headers outside the notebook.")
    current = trace.get_tracer_provider()
    if type(current).__name__ != "ProxyTracerProvider":
        raise RuntimeError("Tracing is already configured; restart the kernel first.")

    with (
        DefaultAzureCredential() as credential,
        AIProjectClient(
            endpoint=session.project_endpoint, credential=credential
        ) as project,
    ):
        # This value is live telemetry configuration: never print or persist it.
        app_insights_value = (
            project.telemetry.get_application_insights_connection_string()
        )
        if "=" in app_insights_value:
            app_insights_connection = app_insights_value
        else:
            # Some linked projects return only the legacy instrumentation-key GUID.
            UUID(app_insights_value)
            resource_id = session.labs.observability.application_insights_resource_id
            if not session.observability_ready:
                raise RuntimeError(
                    "Configure the linked Application Insights resource ID."
                )
            arm_token = credential.get_token(
                "https://management.azure.com/.default"
            ).token
            arm_response = httpx.get(
                f"https://management.azure.com{resource_id}",
                params={"api-version": "2020-02-02"},
                headers={"Authorization": f"Bearer {arm_token}"},
                timeout=30.0,
            )
            arm_response.raise_for_status()
            properties = arm_response.json()["properties"]
            app_insights_connection = properties.get(
                "ConnectionString", properties.get("connectionString", "")
            )
            if not app_insights_connection:
                raise RuntimeError(
                    "ARM returned no Application Insights connection string."
                )

    provider = TracerProvider(
        resource=Resource.create(
            {
                "service.name": "foundry-curriculum-agent",
                "service.version": session.labs.agent.version,
            }
        )
    )
    provider.add_span_processor(
        BatchSpanProcessor(
            AzureMonitorTraceExporter(connection_string=app_insights_connection)
        )
    )
    # OTLPSpanExporter reads endpoint and headers from the external environment.
    provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
    trace.set_tracer_provider(provider)
    instrumentor = AIProjectInstrumentor()
    try:
        instrumentor.instrument(
            enable_content_recording=False,
            enable_trace_context_propagation=True,
            enable_baggage_propagation=False,
        )
    except Exception:
        provider.shutdown()
        raise
    return instrumentor, provider


if RUN_DUAL_EXPORT:
    print(
        {
            "configuration_ready": True,
            "content_recording": False,
            "next_step": "Run one trace in the next cell.",
        }
    )
else:
    print("Connected exporters skipped; configure once in a fresh kernel.")

In [ ]:
RUN_AGENT_TRACE = False

# This is only a dual-export/correlation smoke test, not an Agent Framework lab.
if RUN_AGENT_TRACE:
    from opentelemetry import trace

    if not RUN_DUAL_EXPORT:
        raise RuntimeError("Configure dual export in the previous cell first.")
    connected_instrumentor, connected_provider = configure_connected_dual_export()
    try:
        tracer = trace.get_tracer("foundry-curriculum.agent")
        with tracer.start_as_current_span(
            "assurance.request",
            attributes={
                "aai.instrumentation.owner": "application",
                "aai.agent.id": session.labs.agent.id,
                "aai.agent.name": session.labs.agent.name,
            },
        ) as root_span:
            conversation, response = helpers.create_agent_response(
                session,
                "Return the word ready. This is a synthetic telemetry smoke test.",
                allow_network=True,
            )
            root_span.set_attribute("session.id", conversation.id)
            root_span.set_attribute("gen_ai.conversation.id", conversation.id)
            emitted_trace_id = root_span.get_span_context().trace_id
        connected_provider.force_flush()
        print(
            {
                "trace_id_hex": f"{emitted_trace_id:032x}",
                "conversation_id": conversation.id,
                "response_id": response.id,
                "content_recording": False,
                "test_scope": "dual_export_correlation_smoke",
            }
        )
    finally:
        try:
            connected_instrumentor.uninstrument()
        finally:
            connected_provider.shutdown()
else:
    print("Agent trace skipped; no telemetry or model request was sent.")

In [ ]:
agent_framework_pattern = {
    "certified_core_version": "1.12.1",
    "current_api": "agent_framework.observability.configure_otel_providers",
    "native_subtree": {
        "parent": "invoke_agent",
        "direct_sibling_children": ["chat", "execute_tool", "chat"],
        "hosted_protocol_parent": "live-validate",
        "certified_tool_hierarchy": "authenticated live validation",
    },
    "exporters": ["AzureMonitorTraceExporter", "OTLPSpanExporter"],
    "sensitive_data": False,
    "native_gaps": [
        "decision reason and evidence references",
        "retry or recovery decision",
        "portable human-approval semantic span",
    ],
    "tool_rule": ("model tool-call content is intent; execute_tool is execution"),
    "owner_rule": (
        "use this instead of creating a second global instrumentation owner"
    ),
}
agent_framework_pattern

## Verification and exit criteria

First prove that both exporters received the same trace and parent/span IDs locally. The offline topology is explicitly synthetic and is not a fabricated MLflow trace. The connected path is only a dual-export/correlation smoke test: verify the Application Insights link and viewer RBAC, run one synthetic Foundry Responses request, flush the provider, then wait for ingestion before searching Foundry by trace/conversation ID. The connected `finally` block always un-instruments the Foundry client and shuts down the provider owned by this lab. Independently verify the MLflow receiver. Do not declare success because the model answered; require evidence from both destinations.

Debug from the producer outward: in-memory topology and IDs; one instrumentation owner and its privacy switches; exporter endpoint, protocol, headers, flush, and renewable authentication; Application Insights ingestion and viewer RBAC; Foundry correlation; then MLflow ingestion, managed semantic translation, and independent scorers. The managed root and span-type mapping remain live-validation items. The certified Agent Framework `invoke_agent` / `chat` / `execute_tool` hierarchy remains a separate authenticated live validation with a real tool-enabled Agent Framework application; this notebook does not invoke Agent Framework.

For production Databricks trace storage, select the Unity Catalog location when the experiment is created, confirm SQL warehouse and table permissions, and account for current ingestion and Private Link limitations. MLflow remains the authoritative assurance evidence plane. Select useful production traces, minimize sensitive content, obtain human review and Feedback, promote only approved cases into a versioned EvaluationDataset, and rerun the ordinary gate. Do not infer retry, recovery, or human intervention from a good answer when the observed execution does not record it.